In [2]:
import cv2
import json
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from datetime import datetime

# ==========================================================
# CONFIG
# ==========================================================

VIDEO_DIR = Path("../dataset/raw_videos")
OUTPUT_DIR = Path("../output")

OUTPUT_DIR.mkdir(exist_ok=True)

GLOBAL_LOG = OUTPUT_DIR / "preprocessing.log"

metadata_csv = []


# ==========================================================
# Global Logger
# ==========================================================

def global_log(message):

    print(message)

    with open(GLOBAL_LOG, "a", encoding="utf-8") as f:
        f.write(message + "\n")


global_log("=" * 70)
global_log(f"Start : {datetime.now()}")
global_log("=" * 70)

# ==========================================================
# Read Videos
# ==========================================================

video_files = sorted(VIDEO_DIR.glob("*"))

global_log(f"พบทั้งหมด {len(video_files)} วิดีโอ\n")

# ==========================================================
# Process
# ==========================================================

for video_path in video_files:

    print()
    global_log("-" * 70)
    global_log(f"Reading : {video_path.name}")

    # ======================================================
    # Create Folder
    # ======================================================

    video_name = video_path.stem

    video_output = OUTPUT_DIR / video_name

    preview_dir = video_output / "preview"

    preview_dir.mkdir(parents=True, exist_ok=True)

    log_file = video_output / "log.txt"

    metadata_file = video_output / "metadata.json"

    def log(msg):

        print(msg)

        with open(log_file, "a", encoding="utf-8") as f:

            f.write(msg + "\n")

    # ======================================================
    # Read Video
    # ======================================================

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():

        log("Cannot open video")

        continue

    fps = cap.get(cv2.CAP_PROP_FPS)

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    duration = frame_count / fps if fps > 0 else 0

    status = "READY"

    if fps < 20:

        status = "LOW_FPS"

    if width < 640:

        status = "LOW_RESOLUTION"

    # ======================================================
    # Sample Frames
    # ======================================================

    sample_index = [

        0,

        frame_count // 4,

        frame_count // 2,

        frame_count * 3 // 4,

        max(frame_count - 1, 0)

    ]

    frames = []

    for idx in sample_index:

        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)

        ret, frame = cap.read()

        if ret:

            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            frames.append((idx, frame))

    cap.release()

    # ======================================================
    # Save Preview
    # ======================================================

    preview_path = preview_dir / "preview.jpg"

    fig = plt.figure(figsize=(18,4))

    for i, (idx, frame) in enumerate(frames):

        ax = plt.subplot(1, len(frames), i+1)

        ax.imshow(frame)

        ax.set_title(f"Frame {idx}")

        ax.axis("off")

    plt.suptitle(

        f"{video_path.name}\n"

        f"{width}x{height} | "

        f"{fps:.2f} FPS | "

        f"{duration:.2f} sec"

    )

    plt.tight_layout()

    plt.savefig(preview_path, dpi=200)

    plt.close()

    # ======================================================
    # Save metadata.json
    # ======================================================

    metadata = {

        "video_name": video_path.name,

        "video_path": str(video_path),

        "width": width,

        "height": height,

        "fps": round(fps,2),

        "frame_count": frame_count,

        "duration": round(duration,2),

        "preview": "preview/preview.jpg",

        "status": status,

        "created_at": str(datetime.now())

    }

    with open(metadata_file, "w", encoding="utf-8") as f:

        json.dump(

            metadata,

            f,

            indent=4,

            ensure_ascii=False

        )

    # ======================================================
    # Save CSV
    # ======================================================

    metadata_csv.append({

        "video": video_name,

        "fps": round(fps,2),

        "width": width,

        "height": height,

        "frame_count": frame_count,

        "duration": round(duration,2),

        "status": status

    })

    # ======================================================
    # Log
    # ======================================================

    log(f"Video : {video_name}")

    log(f"Resolution : {width}x{height}")

    log(f"FPS : {fps:.2f}")

    log(f"Frames : {frame_count}")

    log(f"Duration : {duration:.2f}")

    log(f"Status : {status}")

    log("Preview Saved")

    log("Metadata Saved")

    global_log(f"{video_name} : DONE")

# ==========================================================
# metadata.csv
# ==========================================================

df = pd.DataFrame(metadata_csv)

df.to_csv(

    OUTPUT_DIR / "metadata.csv",

    index=False,

    encoding="utf-8-sig"

)

global_log("\nmetadata.csv saved")

global_log("Finished")

Start : 2026-08-16 15:45:45.870990
พบทั้งหมด 3 วิดีโอ


----------------------------------------------------------------------
Reading : video001.webm
Video : video001
Resolution : 1920x1080
FPS : 25.00
Frames : 43726
Duration : 1749.04
Status : READY
Preview Saved
Metadata Saved
video001 : DONE

----------------------------------------------------------------------
Reading : video002.webm
Video : video002
Resolution : 1920x1080
FPS : 25.00
Frames : 39501
Duration : 1580.04
Status : READY
Preview Saved
Metadata Saved
video002 : DONE

----------------------------------------------------------------------
Reading : video003.webm
Video : video003
Resolution : 1920x1080
FPS : 30.00
Frames : 42625
Duration : 1420.83
Status : READY
Preview Saved
Metadata Saved
video003 : DONE

metadata.csv saved
Finished
